In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_ecobici_dev")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsmartdataproy26")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

In [0]:
base = f"abfss://raw@{storageName}.dfs.core.windows.net/datasets"
ruta_origen   = f"{base}/estaciones"
tabla_destino = f"{catalogo}.bronze.estaciones_raw"

In [0]:
archivos = [f.name for f in dbutils.fs.ls(ruta_origen) if not f.name.startswith("_")]
print(archivos)
assert len(archivos) >= 1

['caracteristicas_cicloestaciones.json']


In [0]:
archivo = archivos[0]
df = (
    spark.read
    .option("multiLine", "true")
    .json(f"{ruta_origen}/{archivo}")
)
df.printSchema()

root
 |-- fields: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- type: string (nullable = true)
 |-- records: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: string (containsNull = true)



In [0]:
display(
    df.select(
        size(col("fields")).alias("num_fields"),
        size(col("records")).alias("num_records")
    )
)

num_fields,num_records
11,677


In [0]:
df.selectExpr("fields").show(truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|fields                                                                                                                                                                                              |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[{_id, int}, {sistema, text}, {num_cicloe, text}, {calle_prin, text}, {calle_secu, text}, {colonia, text}, {alcaldia, text}, {latitud, text}, {longitud, text}, {sitio_de_e, text}, {estatus, text}]|
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+



In [0]:
columnas = [
    x["id"]
    for x in df.select("fields").first()["fields"]
]

print(columnas)

['_id', 'sistema', 'num_cicloe', 'calle_prin', 'calle_secu', 'colonia', 'alcaldia', 'latitud', 'longitud', 'sitio_de_e', 'estatus']


In [0]:
registros = df.select(explode("records").alias("r"))

estaciones_bronze = registros.select(
    *[
        col("r")[i].alias(columnas[i])
        for i in range(len(columnas))
    ]
)

display(estaciones_bronze)

_id,sistema,num_cicloe,calle_prin,calle_secu,colonia,alcaldia,latitud,longitud,sitio_de_e,estatus
1,Ecobici,423,Moras,Eje 8 Sur Jose Maria Rico,Acacias,Benito Juarez,19.367816,-99.175269,Arroyo,Instalada
2,Ecobici,437,Moras,Comunal,Acacias,Benito Juarez,19.364916,-99.175985,Arroyo,Instalada
3,Ecobici,438,Adolfo Prieto,Jose Maria Olloqui,Acacias,Benito Juarez,19.36473,-99.174447,Arroyo,Instalada
4,Ecobici,439,Amores,Avenida Coyoacan,Acacias,Benito Juarez,19.362037,-99.172683,Arroyo,Instalada
5,Ecobici,440,Martin Mendalde,M. Lopez Cotilla,Acacias,Benito Juarez,19.361699,-99.171387,Arroyo,Instalada
6,Ecobici,415,San Francisco,Parroquia,Actipan,Benito Juarez,19.37099,-99.176135,Arroyo,Instalada
7,Ecobici,420,Maria Luz Bringas,Oso,Actipan,Benito Juarez,19.369782,-99.179707,Arroyo,Instalada
8,Ecobici,421,Maria Luz Bringas,Oso,Actipan,Benito Juarez,19.36952,-99.179421,Arroyo,Instalada
9,Ecobici,422,Bufalo,Eje 8 Sur Jose Maria Rico,Actipan,Benito Juarez,19.368403,-99.177472,Arroyo,Instalada
10,Ecobici,436,Rodriguez Saro,Bufalo,Actipan,Benito Juarez,19.36623,-99.177904,Arroyo,Instalada


In [0]:
estaciones_bronze = (
    estaciones_bronze
        .withColumn("_archivo_origen", lit(archivo))
        .withColumn("_fecha_ingesta", current_timestamp())
)

display(estaciones_bronze)

_id,sistema,num_cicloe,calle_prin,calle_secu,colonia,alcaldia,latitud,longitud,sitio_de_e,estatus,_archivo_origen,_fecha_ingesta
1,Ecobici,423,Moras,Eje 8 Sur Jose Maria Rico,Acacias,Benito Juarez,19.367816,-99.175269,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:14.561965Z
2,Ecobici,437,Moras,Comunal,Acacias,Benito Juarez,19.364916,-99.175985,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:14.561965Z
3,Ecobici,438,Adolfo Prieto,Jose Maria Olloqui,Acacias,Benito Juarez,19.36473,-99.174447,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:14.561965Z
4,Ecobici,439,Amores,Avenida Coyoacan,Acacias,Benito Juarez,19.362037,-99.172683,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:14.561965Z
5,Ecobici,440,Martin Mendalde,M. Lopez Cotilla,Acacias,Benito Juarez,19.361699,-99.171387,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:14.561965Z
6,Ecobici,415,San Francisco,Parroquia,Actipan,Benito Juarez,19.37099,-99.176135,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:14.561965Z
7,Ecobici,420,Maria Luz Bringas,Oso,Actipan,Benito Juarez,19.369782,-99.179707,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:14.561965Z
8,Ecobici,421,Maria Luz Bringas,Oso,Actipan,Benito Juarez,19.36952,-99.179421,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:14.561965Z
9,Ecobici,422,Bufalo,Eje 8 Sur Jose Maria Rico,Actipan,Benito Juarez,19.368403,-99.177472,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:14.561965Z
10,Ecobici,436,Rodriguez Saro,Bufalo,Actipan,Benito Juarez,19.36623,-99.177904,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:14.561965Z


In [0]:
(
    estaciones_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabla_destino)
)

In [0]:
bronze = spark.table(tabla_destino)
print("Total de estaciones:", bronze.count())
display(bronze.limit(10))

Total de estaciones: 677


_id,sistema,num_cicloe,calle_prin,calle_secu,colonia,alcaldia,latitud,longitud,sitio_de_e,estatus,_archivo_origen,_fecha_ingesta
1,Ecobici,423,Moras,Eje 8 Sur Jose Maria Rico,Acacias,Benito Juarez,19.367816,-99.175269,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:47.842569Z
2,Ecobici,437,Moras,Comunal,Acacias,Benito Juarez,19.364916,-99.175985,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:47.842569Z
3,Ecobici,438,Adolfo Prieto,Jose Maria Olloqui,Acacias,Benito Juarez,19.36473,-99.174447,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:47.842569Z
4,Ecobici,439,Amores,Avenida Coyoacan,Acacias,Benito Juarez,19.362037,-99.172683,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:47.842569Z
5,Ecobici,440,Martin Mendalde,M. Lopez Cotilla,Acacias,Benito Juarez,19.361699,-99.171387,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:47.842569Z
6,Ecobici,415,San Francisco,Parroquia,Actipan,Benito Juarez,19.37099,-99.176135,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:47.842569Z
7,Ecobici,420,Maria Luz Bringas,Oso,Actipan,Benito Juarez,19.369782,-99.179707,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:47.842569Z
8,Ecobici,421,Maria Luz Bringas,Oso,Actipan,Benito Juarez,19.36952,-99.179421,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:47.842569Z
9,Ecobici,422,Bufalo,Eje 8 Sur Jose Maria Rico,Actipan,Benito Juarez,19.368403,-99.177472,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:47.842569Z
10,Ecobici,436,Rodriguez Saro,Bufalo,Actipan,Benito Juarez,19.36623,-99.177904,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:47.842569Z
